In [0]:
CREATE OR REPLACE VIEW t.Date AS
WITH params AS (
  SELECT
    to_date('1980-01-01') AS start_date,
    to_date('2100-12-31') AS end_date
),
dates AS (
  SELECT
    explode(sequence(start_date, end_date, interval 1 day)) AS Date
  FROM
    params
),
base AS (
  SELECT
    Date,
    -- Perioden als DATE (Periodenstart)
    trunc(Date, 'MM') AS MonthDate,
    trunc(Date, 'Q') AS QuarterDate,
    trunc(Date, 'YEAR') AS YearDate,
    -- Numerische Attribute
    year(Date) AS Year,
    month(Date) AS Month,
    quarter(Date) AS Quarter,
    -- Textformate
    date_format(Date, 'MMM') AS MMM,
    date_format(Date, 'EEE') AS DDD,
    -- Sort Keys
    year(Date) AS YearSortKey,
    (year(Date) * 100 + month(Date)) AS YearMonthSortKey,
    (year(Date) * 10 + quarter(Date)) AS YearQuarterSortKey,
    -- Hilfsschlüssel für Offsets
    (year(Date) * 12 + month(Date)) AS YearMonthKey,
    (year(Date) * 4 + quarter(Date)) AS YearQuarterKey
  FROM
    dates
),
curr AS (
  SELECT
    year(current_date()) AS CurrYear,
    (year(current_date()) * 12 + month(current_date())) AS CurrYearMonthKey,
    (year(current_date()) * 4 + quarter(current_date())) AS CurrYearQuarterKey
)
SELECT
  b.Date,
  -- DATE-Spalten für Perioden
  b.MonthDate,
  b.QuarterDate,
  b.YearDate,
  -- Anzeige / Attribute
  b.MMM,
  b.DDD,
  -- Sort Keys
  b.YearSortKey,
  b.YearQuarterSortKey,
  b.YearMonthSortKey,
  -- Offsets
  (b.Year - c.CurrYear) AS YearOffset,
  (b.YearQuarterKey - c.CurrYearQuarterKey) AS QuarterOffset,
  (b.YearMonthKey - c.CurrYearMonthKey) AS MonthOffset
FROM
  base b CROSS JOIN curr c;